# Tutorial 7 — A stock and a stochastic price

**Source:** `../PS4/Problem_Set_4_Q2.py`

**In class:** PS4 Question 2(f); start with the three-state calculation from part (c).

Run cells from top to bottom. At each pause, predict the result before running the next cell. Plots appear below their cells; the source scripts remain the full resolution reference.


## 1. Two state variables

$$V(S,p)=\max_{0\le q<S}\left\{pq-\frac{q^2}{S}-\kappa q^2+\beta\,\mathbb E[V(S-q,p')\mid p]\right\}.$$

$S$ is the remaining stock and $p$ is today's price. Extraction $q$ is the control. The price follows a persistent Markov process.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm


In [ ]:
BETA, RHO, SIGMA, MEAN_PRICE = .95, .8, .10, 2.0
N_P, N_S, N_Q = 5, 120, 60  # live grids; source uses 7, 600, 300


## 2. Replace the continuous price shock with a few states

Tauchen's method puts nodes across the stationary range of log price. Each row of $P$ contains probabilities of tomorrow's price conditional on today's price. The rows must sum to one.


In [ ]:
def tauchen(rho, sigma, n, m=3.0, mu=0.0):
    """
    Discretise log z' = rho log z + sigma eps' onto an n-state Markov chain.

    Nodes are evenly spaced across +/- m unconditional standard deviations.
    The two outer columns absorb the whole remaining tail, which is what makes
    each row sum to one.
    """
    sd = sigma / np.sqrt(1 - rho**2)
    z = np.linspace(-m * sd, m * sd, n)
    w = z[1] - z[0]
    P = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if j == 0:
                P[i, j] = norm.cdf((z[0] - rho * z[i] + w / 2) / sigma)
            elif j == n - 1:
                P[i, j] = 1 - norm.cdf((z[-1] - rho * z[i] - w / 2) / sigma)
            else:
                P[i, j] = norm.cdf((z[j] - rho * z[i] + w / 2) / sigma) - norm.cdf(
                    (z[j] - rho * z[i] - w / 2) / sigma
                )
    return np.exp(z + mu), P


In [ ]:
p3, P3 = tauchen(RHO, SIGMA, 3)
print('Price nodes:', np.round(p3, 4))
print('Transition probabilities:', np.round(P3, 4))
print('Row sums:', P3.sum(axis=1))


## 3. Why homogeneity may hold

With $\kappa=0$, replacing $(S,q)$ by $(\lambda S,\lambda q)$ multiplies current payoff and next stock by $\lambda$. Thus $V(S,p)=a(p)S$ and $q/S$ should not depend on $S$. The term $\kappa q^2$ scales by $\lambda^2$ and breaks this result.


In [ ]:
p, P = tauchen(RHO, SIGMA, N_P, mu=np.log(MEAN_PRICE))
S = np.exp(np.linspace(np.log(1e-3), np.log(5), N_S))
frac = np.linspace(0, .99, N_Q)
print('State grid shape (stock, price):', (len(S), len(p)))
print('Choice grid: extraction fractions from', frac[0], 'to', frac[-1])


## 4. One expected-value calculation

If $V$ is a stock-by-price matrix, `V @ P.T` gives expected continuation value at each stock conditional on today's price. For a chosen extraction fraction, the next stock $S-q$ normally falls between nodes, so we interpolate.


In [ ]:
V_guess = S[:, None] * np.ones((1, N_P))
EV = V_guess @ P.T
si, pi, fi = N_S//2, N_P//2, N_Q//2
q_example = S[si]*frac[fi]
continuation = np.interp(S[si]-q_example, S, EV[:, pi])
print('stock, chosen extraction, expected next value:', S[si], q_example, continuation)


## 5. Iterate the Bellman equation

The complete solver evaluates every extraction fraction at every $(S,p)$, selects the best, and repeats. Inspect the matrix dimensions in the function before running it.


In [ ]:
def solve(kappa, tol=1e-9, maxit=4000):
    V = np.zeros((N_S, N_P))
    Q = np.zeros((N_S, N_P))
    q = S[:, None] * frac[None, :]                       # (N_S, N_Q)
    S_next = S[:, None] * (1 - frac)[None, :]
    flow = (
        p[None, None, :] * q[:, :, None]
        - (q**2 / S[:, None])[:, :, None]
        - kappa * (q**2)[:, :, None]
    )
    for it in range(maxit):
        EV = V @ P.T                                     # EV[s, i] = E[V(S_s, p') | p_i]
        cont = np.empty((N_S, N_Q, N_P))
        for i in range(N_P):
            cont[:, :, i] = np.interp(S_next, S, EV[:, i])
        val = flow + BETA * cont
        k = val.argmax(axis=1)
        V_new = np.take_along_axis(val, k[:, None, :], axis=1)[:, 0, :]
        Q_new = q[np.arange(N_S)[:, None], k]
        gap = np.max(np.abs(V_new - V))
        V, Q = V_new, Q_new
        if gap < tol:
            break
    return V, Q, it + 1, gap


In [ ]:
V0, Q0, it0, gap0 = solve(kappa=0.0)
print(f'No capacity cost: {it0} iterations; gap {gap0:.2e}')


## 6. Add capacity cost

Now an additional unit of stock can make proportional extraction expensive. Predict which curve will vary with stock.


In [ ]:
V1, Q1, it1, gap1 = solve(kappa=0.5)
print(f'Capacity cost 0.5: {it1} iterations; gap {gap1:.2e}')


In [ ]:
mid = N_P//2
for target in (.1, .5, 1.5, 3, 4.5):
    i = np.argmin(np.abs(S-target))
    print(f'S={S[i]:.2f}: q/S = {Q0[i,mid]/S[i]:.3f} (κ=0), {Q1[i,mid]/S[i]:.3f} (κ=.5)')


## 7. See the homogeneity test

The left panel should be nearly flat when the capacity cost is zero. Small-stock deviations reflect finite-grid interpolation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for value, extraction, label in ((V0, Q0, 'κ=0'), (V1, Q1, 'κ=.5')):
    axes[0].plot(S, value[:,mid]/S, label=label)
    axes[1].plot(S, extraction[:,mid]/S, label=label)
axes[0].set_title('Value per unit of stock')
axes[1].set_title('Extraction fraction')
for ax in axes:
    ax.set_xlabel('$S$'); ax.set_xlim(0, 5); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
